# HotpotQA bridge questions under the mask protocol

The MuSiQue mask study on a second dataset. 200 HotpotQA bridge questions were frozen in the repo before any answer (`experiments/wmac_hotpot_v1/prepared.json`). This notebook runs them through the six conditions of the MuSiQue main run and its three shortcut probes, with the same models, prompts, decoding and scoring.

HotpotQA does not label hop order or a bridge entity. The supporting paragraph that holds the answer is treated as hop 2, and the bridge is a supporting paragraph's title that the other supporting paragraph mentions. HotpotQA ships 10 paragraphs per question, so each worker holds 5 (MuSiQue: 10). The rules are in `latent_dispatch/wmac_hotpot.py`.

Pick an **A100 (High-RAM)**, then *Run all*. An **L4** also works: these contexts are short enough for the 8B models in bfloat16 to fit in its 24 GB. It is slower (my estimate, not measured: about three times, so one 8B model per night), and the MuSiQue 8B runs were on an A100, so the hardware would differ from them. The run prints its real pace after the first 25 answers. Two things need you in the first minute: allow Google Drive, then the Hugging Face token (only if `"llama"` is in `MODELS`; a Colab secret named `HF_TOKEN` is picked up by itself once you allow this notebook to read it).

**Before you leave:** wait for the line `ready: nothing below waits on you` (about 5 minutes in).

**What it runs, per model:** `probes` (closed book, and each supporting paragraph alone with all distractors, AB layouts: 600 answers), then `main` (blocked, gold, full, equal-token window, joint reading and stitched caches on the AB and BA layout of every source: 2,400 answers).

**Time and compute units** (estimates, not measured on HotpotQA with these models: the MuSiQue run took 3.7 s per answer on Qwen3-8B and 4.1 s on Llama, and HotpotQA answers took 0.74 of the MuSiQue time in an earlier run on Qwen3-4B; A100 High-RAM at 6.73 units per hour): about 2.3 hours for Qwen3-8B, 2.5 hours for Llama and 1.5 hours for Qwen3-4B, together about 6.3 hours and 42 units, plus model downloads. The models run in that order, so the two 8B models finish first. Qwen3-4B runs in float16, the precision of its MuSiQue run, whatever the GPU; its MuSiQue run was on an L4, so its hardware differs here. To leave a model out, delete it from `MODELS`.

**What goes to Drive** (folder `latent-dispatch-wmac-hotpot`), every 5 minutes and when each part ends: `<model>/<part>/rows.jsonl`, `plan.json`, `runtime.json`, `status.json`, and `log.txt`.

**If the session dies:** open this notebook on a new machine and *Run all*. It pulls back what is in Drive and continues; answers already saved are not redone.


In [ ]:
REPO, COMMIT = "https://github.com/JoshBong/latent-dispatch.git", "632ab82f0ddfd66ce30416cce4cd781073635a65"
ROOT = "/content/latent-dispatch-wmac-hotpot-code"                     # this repo at that commit
BACKUP = "/content/drive/MyDrive/latent-dispatch-wmac-hotpot"          # everything this notebook saves
OUT = "/content/wmac-hotpot"                                           # results on this machine
MODELS = ["8b", "llama", "4b"]           # run in this order; delete a name to leave that model out
PARTS = ["probes", "main"]               # per model: the shortcut probes, then the six conditions
SOURCES = 200                            # the first N sources of the frozen order (200 = all)
SNAPSHOT_MINUTES = 5
LLAMA = "meta-llama/Llama-3.1-8B-Instruct"


In [ ]:
# The two steps that need you: Google Drive, then (for Llama) the Hugging Face token.
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(BACKUP, exist_ok=True)

if "llama" in MODELS:
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face Read token: ").strip()
    os.environ["HF_TOKEN"] = token
    from huggingface_hub import hf_hub_download
    hf_hub_download(LLAMA, "config.json")   # fails here, not hours in, if this account is not approved for Llama
    print("Llama access OK")


In [ ]:
import collections, hashlib, json, os, shutil, subprocess, threading, time

os.environ["TZ"] = "America/New_York"
time.tzset()


def sh(cmd):
    """Run a command, show its output as it comes, return its exit code."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()


def must(cmd):
    if sh(cmd):
        raise RuntimeError(f"failed: {cmd}")


def note(message):
    """Print one line and add it to log.txt in Drive."""
    line = time.strftime("%m-%d %H:%M:%S ") + message
    print(line, flush=True)
    try:
        with open(f"{BACKUP}/log.txt", "a") as f:
            f.write(line + "\n")
    except Exception as error:
        print("  (log.txt not written:", type(error).__name__, error, ")", flush=True)


def put(source, name):
    """Copy one file into the Drive folder. A half-written copy never replaces a good one."""
    target, partial = f"{BACKUP}/{name}", f"{BACKUP}/{name}.uploading"
    os.makedirs(os.path.dirname(target), exist_ok=True)
    shutil.copyfile(source, partial)
    try:
        os.replace(partial, target)
    except OSError:
        shutil.copyfile(source, target)
        os.remove(partial)


def results(folder):
    """The small result files under a folder, as paths relative to it."""
    found = []
    for base, _, names in os.walk(folder):
        for name in sorted(names):
            if name.endswith((".jsonl", ".json")) and name != "log.txt":
                found.append(os.path.relpath(os.path.join(base, name), folder))
    return sorted(found)


def backup():
    """Copy every result file to Drive. Safe while a part is running: a row cut in half is redone on restart."""
    for name in results(OUT):
        put(f"{OUT}/{name}", name)
    return len(results(OUT))


def restore():
    """On a new machine: bring back what Drive has, so saved answers are not redone."""
    if results(OUT):
        return 0
    names = results(BACKUP)
    for name in names:
        os.makedirs(os.path.dirname(f"{OUT}/{name}"), exist_ok=True)
        shutil.copyfile(f"{BACKUP}/{name}", f"{OUT}/{name}")
    return len(names)


def saved(model, part):
    path = f"{OUT}/{model}/{part}/rows.jsonl"
    if not os.path.exists(path):
        return 0
    with open(path, "rb") as f:
        return sum(1 for line in f if line.endswith(b"\n"))


def watch(stop):
    while not stop.wait(SNAPSHOT_MINUTES * 60):
        try:
            backup()
        except Exception as error:
            print(f"[backup] failed, will retry: {type(error).__name__}: {error}", flush=True)


In [ ]:
import torch

assert torch.cuda.is_available(), "no GPU: pick an A100 or an L4"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(torch.cuda.get_device_name(0), f"{gb:.0f} GB")
if set(MODELS) & {"8b", "llama"}:
    assert gb > 20, "the 8B models in bfloat16 need more than 20 GB: pick an A100 or an L4"
if not os.path.exists(ROOT):
    must(f"rm -rf {ROOT}.tmp && git clone -q {REPO} {ROOT}.tmp && git -C {ROOT}.tmp checkout -q {COMMIT} && mv {ROOT}.tmp {ROOT}")
os.chdir(ROOT)
must("git log -1 --format='running commit %h (%s)'")
must("pip install -q -e '.[dev]'")


In [ ]:
os.chdir(ROOT)
must("python -m pytest tests/test_wmac_hotpot.py -q")                    # planner and runner on tiny random models (CPU)
print("question set sha256:", hashlib.sha256(open("experiments/wmac_hotpot_v1/prepared.json", "rb").read()).hexdigest())
must(f"python scripts/run_wmac_hotpot.py preflight --model {','.join(MODELS)}")   # tokenizers only: same token sequences as at preparation
probe = f"{BACKUP}/.write-test"
with open(probe, "w") as f:
    f.write("ok")
assert open(probe).read() == "ok", "the Drive folder did not read back what was written"
os.remove(probe)
print("restored from Drive:", restore(), "files")
for model in MODELS:
    for part in PARTS:
        print(f"  {model:6s} {part:7s} {saved(model, part):5d} rows already saved")
note("ready: nothing below waits on you")


In [ ]:
os.chdir(ROOT)
status = {}
stop = threading.Event()
threading.Thread(target=watch, args=(stop,), daemon=True).start()
try:
    for model in MODELS:
        for part in PARTS:
            cmd = f"python scripts/run_wmac_hotpot.py run --model {model} --part {part} --out {OUT} --sources {SOURCES}"
            print(f"\n===== {model} {part} =====", flush=True)
            code = sh(cmd)
            if code != 0:                                 # one more try; it continues from the saved rows
                print(f"\n===== {model} {part}, second try =====", flush=True)
                code = sh(cmd)
            status[f"{model} {part}"] = code
            try:
                backup()
                note(f"{model} {part}: {'finished' if code == 0 else 'NOT finished (exit code ' + str(code) + ')'}, "
                     f"{saved(model, part)} rows in Drive")
            except Exception as error:
                note(f"{model} {part}: exit code {code}, but the copy to Drive failed ({type(error).__name__}: {error})")
finally:
    stop.set()
print(status)


In [ ]:
# A first look: exact match and F1 per condition and the paired contrasts, on sources that have every answer.
os.chdir(ROOT)
sh(f"python scripts/run_wmac_hotpot.py summary --out {OUT}")
try:
    backup()
except Exception as error:
    note(f"last copy to Drive failed ({type(error).__name__}: {error})")
note("done: " + ", ".join(f"{name} {'finished' if code == 0 else 'NOT finished'}" for name, code in status.items()))
try:
    drive.flush_and_unmount()
    print("Drive is up to date. Safe to disconnect and delete the runtime.")
except Exception as error:
    print("could not confirm the Drive sync:", type(error).__name__, error)
